# Image Generation Models

**Module:** 17 — Image Generation

OpenAI, Stability, FLUX, Imagen, Midjourney, and open checkpoints — selection and integration.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare major providers on control, quality, licensing
- Map tasks to model strengths
- Design a provider-abstracted integration layer
- Build a bakeoff scorecard


## Provider landscape

| Line | Notes | Strengths |
|------|-------|-----------|
| **OpenAI Images** | API-first, safety stack | Integration, general quality |
| **Stability AI** | SD family + APIs | Open ecosystem, ControlNet/LoRA |
| **BFL FLUX** | Modern open/commercial | Prompt adherence, sharpness |
| **Google Imagen** | Cloud / Gemini | Photoreal + GCP |
| **Midjourney** | Creative community | Opinionated aesthetics |
| **Open models** | SDXL, SD3, FLUX-dev… | Self-host, fine-tune, air-gap |

```mermaid
flowchart TB
  APP[App] --> ABS[Provider Interface]
  ABS --> OAI[OpenAI]
  ABS --> STB[Stability]
  ABS --> FLUX[FLUX]
  ABS --> LOC[Self-hosted]
```


## Selection Dimensions

### Definition
Choose along quality, controllability, latency, cost, licensing, safety, and edit surface.

### Why it matters
No universal winner — only winners per job.

### How it works
Fixed golden suite → rubrics → p95 latency → $/100 images → ToS review.

### Intuition
Hiring: portfolio + salary + toolchain fit.

### Pitfalls
- Twitter screenshots alone
- Ignoring commercial ToS
- No fallback provider

### When to use
Before locking architecture or annual contracts.


In [ ]:
# Demo 1: provider interface mocks
from typing import Protocol

class ImageProvider(Protocol):
    name: str
    def generate(self, prompt: str, **kw) -> dict: ...

class MockOpenAI:
    name = "openai"
    def generate(self, prompt: str, **kw) -> dict:
        return {"provider": self.name,
                "request": {"model": kw.get("model", "gpt-image-1"), "prompt": prompt, "size": kw.get("size", "1024x1024")},
                "response": {"data": [{"b64_json": "<...>"}]}}

class MockStability:
    name = "stability"
    def generate(self, prompt: str, **kw) -> dict:
        return {"provider": self.name,
                "request": {"prompt": prompt, "cfg_scale": kw.get("cfg", 7), "engine": "sdxl"},
                "response": {"artifacts": [{"base64": "<...>", "finishReason": "SUCCESS"}]}}

print(MockOpenAI().generate("red cube")["request"])
print(MockStability().generate("red cube")["request"])


In [ ]:
# Demo 2: bakeoff scorecard
W = {"adherence": 0.3, "aesthetics": 0.25, "text": 0.1, "edit_api": 0.15, "cost": 0.1, "license_fit": 0.1}

def score(row): return sum(row[k] * w for k, w in W.items())

board = {
    "openai": {"adherence": 0.9, "aesthetics": 0.85, "text": 0.8, "edit_api": 0.8, "cost": 0.6, "license_fit": 0.9},
    "flux": {"adherence": 0.92, "aesthetics": 0.88, "text": 0.75, "edit_api": 0.5, "cost": 0.7, "license_fit": 0.8},
    "sdxl_selfhost": {"adherence": 0.75, "aesthetics": 0.8, "text": 0.55, "edit_api": 0.95, "cost": 0.9, "license_fit": 0.85},
}
print(sorted(((k, round(score(v), 3)) for k, v in board.items()), key=lambda x: -x[1]))


### Model notes (deepened)

- **OpenAI** — API ergonomics; org safety; `YOUR_OPENAI_API_KEY`
- **Stability** — SDXL/SD3; rich open tooling
- **FLUX** — competitive adherence; check license variants
- **Imagen** — GCP/Vertex fit; region/data controls
- **Midjourney** — high aesthetic floor; verify enterprise/API needs
- **Open weights** — max control; you own MLOps

**Tips:** surgical inpaint → self-host SD-family; managed safety → hosted API; air-gap → open weights.


In [ ]:
# Demo 3: cost estimator
def cost_per_day(images_per_day: int, price_per_image: float, fail_rate: float = 0.08) -> dict:
    attempts = images_per_day / max(1e-6, (1 - fail_rate))
    return {"billable_attempts": round(attempts, 1), "usd": round(attempts * price_per_image, 2)}

print(cost_per_day(5000, 0.04))
print(cost_per_day(5000, 0.02, fail_rate=0.15))


In [ ]:
# Demo 4: FLUX-like HTTP shape
FLUX_API_KEY = "YOUR_BFL_API_KEY"
flux_request = {"prompt": "isometric tiny bakery, soft dawn light", "width": 1024, "height": 1024, "steps": 28, "guidance": 3.5, "seed": 123}
flux_response = {"id": "gen_abc", "status": "ready", "result": {"sample": "https://example.invalid/sample.png"}}
print(FLUX_API_KEY[:8], flux_request["prompt"][:32], flux_response["status"])


### Try it yourself — Model selection

1. Fill the scorecard for an ecommerce use case.
2. Normalize OpenAI and Stability responses to {url|b64, seed, model}.
3. List ToS questions for legal before a campaign.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `bakeoff` | Controlled comparison on a fixed eval suite |
| `open weights` | Downloadable parameters you can self-host |
| `managed API` | Hosted inference with provider SLAs/safety |
| `finishReason` | Success/filter/truncation field |


### Workshop — Parameter journal — Image Models

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Models
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Models

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Models
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Models

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Models
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Models

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Models
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Models

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Models
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Models

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Models
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Image Models

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Image Models
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Image Models

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Image Models
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Abstract providers behind an interface
- Scorecards beat anecdote
- License + safety + edit surface matter as much as demos
